In [7]:
from datetime import date
from enum import Enum
from typing import Self


In [ ]:
from pydantic import BaseModel, EmailStr, ConfigDict, Field, field_validator, model_validator

In [ ]:
class Depart(Enum):
    HR="HR"
    SALES="SALES"
    IT = "IT"
    ENGINEER="ENGINEER"

class Employee(BaseModel):
    model_config = ConfigDict(populate_by_name=True)
    
    e_id: UUID = Field(default_factory=uuid4, frozen=True)
    name: str = Field(min_length=1, max_length=12, frozen=True)
    email: EmailStr = Field(pattern=r".+@ilm\.com$", frozen=True)
    DOB: date = Field(alias="birth_date", repr=False, frozen=True)
    salary: float = Field(alias="Compensation", gt=0, frozen=True)
    department: Depart
    elected_benefit: bool

    @field_validator("DOB")
    @classmethod
    def check_age(cls, DOB: date) -> date:
        today = date.today()
        eighteenyrago= date(today.year-18, today.month, today.day)
        if DOB > eighteenyrago:
            raise ValueError("Employee must by above 18")
        return DOB


    @model_validator(mode="after")
    def check_it_benefit(self) -> Self:
        department = self.department
        electedbenefit = self.elected_benefit
        if department == Depart.IT and electedbenefit:
            raise ValueError("IT employee are not qualify for elected benefit")
        return self

In [ ]:
try:
    emps=Employee(name="Shafin",email="shafi@ilm.com", DOB="1984-10-10", salary=23_22.22, department="IT",elected_benefit=True)
except Exception as e:
    print(e)

In [ ]:
emps

In [8]:
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator
import re


class Coupon(BaseModel):
    """A discount coupon. `discount_percent` must be a multiple of 5."""

    model_config = ConfigDict(populate_by_name=True)
    
    code: str = Field(pattern=r"SAVE([0-9][0-9]?|100)$")
    discount_percent: int = Field(gt=4)
    
    @field_validator("discount_percent")
    @classmethod
    def discount_multiple_of_5(cls, discount_percent: int) -> int:
        if discount_percent%5 !=0:
            raise ValueError("discount_percent must be a multiple of 5")
        return discount_percent
    
    @model_validator(mode="after")
    def code_discount_match(self) -> Self:
        prcnt = self.discount_percent
        cde = self.code
        cde_int = int(re.search(r'\d+', cde).group())
        if cde_int != prcnt:
            raise ValueError("code and discount mismatch")

        return self

In [11]:
try:
    discs=Coupon(code="SAVE10", discount_percent=15)
except Exception as e:
    print(e)

1 validation error for Coupon
  Value error, code and discount mismatch [type=value_error, input_value={'code': 'SAVE10', 'discount_percent': 15}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [10]:
discs

Coupon(code='SAVE10', discount_percent=10)